# ☁️ Ollama Cloud Runner — Google Colab GPU Backend

Run this notebook on **Google Colab** (with a **T4**, **L4**, or **A100** GPU runtime) to power your **Ollama Cloud Runner** web app over **ngrok**.

### Quick Setup
1. In Colab, click **Runtime → Change runtime type → T4 GPU** and click **Save**.
2. Get a free authtoken from [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken) and either:
   - Add it to **Colab Secrets** (🔑 icon on the left sidebar) named `NGROK_AUTHTOKEN`, **or**
   - Paste it when prompted in **Step 3** below.
3. Run all cells below, copy the printed **`https://....ngrok-free.app`** URL, and paste it into the **Ollama Cloud Runner** web app!

In [ ]:
# @title Step 1: Verify GPU & Install Backend Dependencies
!nvidia-smi || echo "⚠️ No GPU detected! Switch Runtime -> Change runtime type -> T4 GPU"
!apt-get update -qq && apt-get install -y -qq zstd pciutils lshw >/dev/null 2>&1 || true
!pip install -q fastapi "uvicorn[standard]" pyngrok httpx psutil beautifulsoup4 pydantic
print("✅ Dependencies installed!")

In [ ]:
# @title Step 2: Bootstrap Modular Backend Server Package
import os
from pathlib import Path

PROJECT_ROOT = Path("/content/ollama_cloud_runner")
SERVER_PKG = PROJECT_ROOT / "server"
SERVER_PKG.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)

(SERVER_PKG / "__init__.py").write_text('"""Colab Backend Package."""\n')

(SERVER_PKG / "config.py").write_text('''import os, sys
from pathlib import Path

IS_LINUX = sys.platform.startswith("linux")
IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
RUNNER_DIR = Path(os.environ.get("OLLAMA_RUNNER_DIR", "/content/ollama_runner"))
RUNNER_DIR.mkdir(parents=True, exist_ok=True)
RUNNER_SCRIPT_PATH = RUNNER_DIR / "start_ollama.sh"
RUNNER_CONFIG_PATH = RUNNER_DIR / "runner_config.json"
OLLAMA_LOG_PATH = RUNNER_DIR / "ollama_server.log"
OLLAMA_HOST = os.environ.get("OLLAMA_INTERNAL_HOST", "http://127.0.0.1:11434")
BACKEND_PORT = int(os.environ.get("PORT", "8000"))
''')

(SERVER_PKG / "models.py").write_text('''from typing import List, Optional, Literal
from pydantic import BaseModel, Field

class GpuMetrics(BaseModel):
    available: bool = False
    name: str = "No GPU Detected (CPU Runtime)"
    vram_total_mb: int = 0
    vram_used_mb: int = 0
    utilization_percent: int = 0
    driver_version: Optional[str] = None

class SystemMetrics(BaseModel):
    cpu_percent: float = 0.0
    ram_total_gb: float = 0.0
    ram_used_gb: float = 0.0
    disk_free_gb: float = 0.0
    disk_total_gb: float = 0.0
    platform_name: str = ""
    is_colab: bool = True

class InstalledModel(BaseModel):
    name: str
    size_bytes: int = 0
    size_gb: float = 0.0
    modified_at: Optional[str] = None
    family: Optional[str] = None
    parameter_size: Optional[str] = None
    quantization_level: Optional[str] = None

class ServerStatusResponse(BaseModel):
    connected: bool = True
    state: Literal["idle", "configuring", "starting", "pulling", "ready", "error"] = "idle"
    state_message: str = "Colab agent connected"
    ollama_installed: bool = False
    ollama_running: bool = False
    ollama_version: Optional[str] = None
    configured_ollama_version: str = "latest"
    active_model: Optional[str] = None
    loaded_models: List[str] = Field(default_factory=list)
    installed_models: List[InstalledModel] = Field(default_factory=list)
    gpu: GpuMetrics
    system: SystemMetrics
    ngrok_url: Optional[str] = None
    uptime_seconds: int = 0
    last_updated_script_at: Optional[str] = None

class DeployRequest(BaseModel):
    model: str
    ollama_version: str = "latest"
    num_ctx: int = 4096
    keep_alive: str = "15m"
    flash_attention: bool = True
    restart_server: bool = True

class ChatMessage(BaseModel):
    role: Literal["system", "user", "assistant"]
    content: str

class ChatRequest(BaseModel):
    model: Optional[str] = None
    messages: List[ChatMessage]
    temperature: float = 0.7
    num_ctx: int = 4096
    stream: bool = True
''')

(SERVER_PKG / "gpu_monitor.py").write_text('''import platform, shutil, subprocess, psutil
from .config import IS_COLAB, RUNNER_DIR
from .models import GpuMetrics, SystemMetrics

def get_gpu_metrics() -> GpuMetrics:
    try:
        res = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,memory.used,utilization.gpu,driver_version", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=3, check=False
        )
        if res.returncode == 0 and res.stdout.strip():
            parts = [p.strip() for p in res.stdout.strip().splitlines()[0].split(",")]
            if len(parts) >= 5:
                return GpuMetrics(available=True, name=parts[0], vram_total_mb=int(float(parts[1])), vram_used_mb=int(float(parts[2])), utilization_percent=int(float(parts[3])), driver_version=parts[4])
    except Exception:
        pass
    return GpuMetrics(available=False)

def get_system_metrics() -> SystemMetrics:
    vm = psutil.virtual_memory()
    disk = shutil.disk_usage(str(RUNNER_DIR) if RUNNER_DIR.exists() else "/")
    return SystemMetrics(cpu_percent=round(psutil.cpu_percent(interval=None), 1), ram_total_gb=round(vm.total / (1024**3), 2), ram_used_gb=round(vm.used / (1024**3), 2), disk_free_gb=round(disk.free / (1024**3), 2), disk_total_gb=round(disk.total / (1024**3), 2), platform_name=f"{platform.system()} {platform.release()}", is_colab=IS_COLAB)
''')

(SERVER_PKG / "catalog_scraper.py").write_text('''import re, time
import httpx
from bs4 import BeautifulSoup

_CACHE = {"ts": 0.0, "models": []}
CURATED = [
    {"name": "llama3", "title": "Llama 3", "provider": "Meta", "category": "general", "featured": True, "description": "Meta Llama 3 — flagship open weights LLM optimized for dialogue and reasoning.", "tags": ["8b", "70b", "instruct"], "default_tag": "8b", "vram_hint": "5.5 GB (8b)", "pulls": "12M+", "capabilities": ["chat", "instruct"]},
    {"name": "llama3.2", "title": "Llama 3.2", "provider": "Meta", "category": "general", "featured": True, "description": "Lightweight 1B and 3B Llama models tailored for fast inference on Colab T4 GPUs.", "tags": ["1b", "3b"], "default_tag": "3b", "vram_hint": "2.4 GB (3b)", "pulls": "9.5M+", "capabilities": ["chat", "tools"]},
    {"name": "llama3.1", "title": "Llama 3.1", "provider": "Meta", "category": "general", "featured": True, "description": "128K context Llama 3.1 with strong tool calling and multilingual support.", "tags": ["8b", "70b"], "default_tag": "8b", "vram_hint": "5.5 GB (8b)", "pulls": "18M+", "capabilities": ["chat", "tools"]},
    {"name": "deepseek-r1", "title": "DeepSeek-R1", "provider": "DeepSeek", "category": "reasoning", "featured": True, "description": "Open reasoning models distilled into Colab-friendly 1.5B, 7B, 8B, and 14B sizes.", "tags": ["1.5b", "7b", "8b", "14b", "32b"], "default_tag": "8b", "vram_hint": "5.5 GB (8b)", "pulls": "22M+", "capabilities": ["reasoning", "chat"]},
    {"name": "gemma3", "title": "Gemma 3", "provider": "Google", "category": "vision", "featured": True, "description": "Google multimodal Gemma 3 family running smoothly on a single Colab T4 GPU.", "tags": ["1b", "4b", "12b", "27b"], "default_tag": "4b", "vram_hint": "3.4 GB (4b)", "pulls": "5.8M+", "capabilities": ["chat", "vision"]},
    {"name": "qwen2.5", "title": "Qwen 2.5", "provider": "Alibaba", "category": "general", "featured": True, "description": "Dense multilingual instruction models from 0.5B to 72B with JSON & tool support.", "tags": ["0.5b", "1.5b", "3b", "7b", "14b"], "default_tag": "7b", "vram_hint": "5.2 GB (7b)", "pulls": "11M+", "capabilities": ["chat", "tools"]},
    {"name": "qwen2.5-coder", "title": "Qwen 2.5 Coder", "provider": "Alibaba", "category": "coding", "featured": True, "description": "Code-specific Qwen 2.5 model excelling at code generation and debugging.", "tags": ["1.5b", "3b", "7b", "14b"], "default_tag": "7b", "vram_hint": "5.2 GB (7b)", "pulls": "6.3M+", "capabilities": ["coding", "tools"]},
    {"name": "mistral", "title": "Mistral 7B", "provider": "Mistral AI", "category": "general", "featured": True, "description": "Fast, reliable 7B model updated to v0.3 with function calling and 32K context.", "tags": ["7b", "latest"], "default_tag": "7b", "vram_hint": "4.8 GB (7b)", "pulls": "14M+", "capabilities": ["chat", "tools"]},
    {"name": "phi4", "title": "Phi-4", "provider": "Microsoft", "category": "reasoning", "featured": True, "description": "Microsoft 14B state-of-the-art compact model for math, STEM, and reasoning.", "tags": ["14b", "latest"], "default_tag": "14b", "vram_hint": "9.5 GB (14b)", "pulls": "3.9M+", "capabilities": ["reasoning", "chat"]}
]

async def fetch_ollama_library_catalog():
    if _CACHE["models"] and (time.time() - _CACHE["ts"]) < 900:
        return _CACHE["models"]
    seen = {m["name"] for m in CURATED}
    merged = [dict(m) for m in CURATED]
    try:
        async with httpx.AsyncClient(timeout=8.0, follow_redirects=True) as client:
            r = await client.get("https://ollama.com/library")
            if r.status_code == 200:
                soup = BeautifulSoup(r.text, "html.parser")
                for a in soup.select('a[href^="/library/"]'):
                    slug = a.get("href", "").replace("/library/", "").strip("/").split("/")[0]
                    if not slug or slug in seen or ":" in slug:
                        continue
                    p = a.find("p")
                    spans = [s.get_text(strip=True).lower() for s in a.find_all("span") if s.get_text(strip=True)]
                    tags = [b for b in spans if re.match(r"^\d+(\.\d+)?[bm]$", b)] or ["latest"]
                    merged.append({"name": slug, "title": slug.replace("-", " ").title(), "provider": "Ollama Library", "category": "general", "featured": False, "description": p.get_text(" ", strip=True) if p else f"Official {slug} model.", "tags": tags[:8], "default_tag": tags[0], "vram_hint": "Auto", "pulls": "Library", "capabilities": ["chat"]})
                    seen.add(slug)
    except Exception:
        pass
    _CACHE["ts"] = time.time()
    _CACHE["models"] = merged
    return merged
''')

(SERVER_PKG / "ollama_manager.py").write_text('''import asyncio, json, os, shutil, subprocess, time
from datetime import datetime, timezone
import httpx
from .config import IS_LINUX, OLLAMA_HOST, OLLAMA_LOG_PATH, RUNNER_CONFIG_PATH, RUNNER_SCRIPT_PATH
from .models import ChatRequest, DeployRequest, InstalledModel

class OllamaManager:
    def __init__(self):
        self.process = None
        self.state = "idle"
        self.state_message = "Colab agent ready — select a model to update & start server"
        self.active_model = None
        self.configured_ollama_version = "latest"
        self.last_updated_script_at = None
        self.started_at = time.time()
        self._write_runner_files(DeployRequest(model="llama3:8b"), initial=True)

    def _write_runner_files(self, req: DeployRequest, initial: bool = False):
        now_iso = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
        cfg = {"model": req.model, "ollama_version": req.ollama_version, "num_ctx": req.num_ctx, "keep_alive": req.keep_alive, "flash_attention": req.flash_attention, "updated_at": now_iso}
        RUNNER_CONFIG_PATH.write_text(json.dumps(cfg, indent=2))
        ver_env = f'OLLAMA_VERSION="{req.ollama_version}" ' if req.ollama_version != "latest" else ""
        script = f"#!/usr/bin/env bash\nset -euo pipefail\nexport OLLAMA_HOST=127.0.0.1:11434\nexport OLLAMA_ORIGINS='*'\nexport OLLAMA_KEEP_ALIVE='{req.keep_alive}'\nexport OLLAMA_FLASH_ATTENTION={'1' if req.flash_attention else '0'}\nif ! command -v ollama >/dev/null 2>&1; then\n  curl -fsSL https://ollama.com/install.sh | {ver_env}sh\nfi\nnohup ollama serve > {OLLAMA_LOG_PATH.as_posix()} 2>&1 &\nsleep 2\nollama pull '{req.model}'\n"
        RUNNER_SCRIPT_PATH.write_text(script)
        os.chmod(RUNNER_SCRIPT_PATH, 0o755)
        if not initial:
            self.last_updated_script_at = now_iso
            self.configured_ollama_version = req.ollama_version
        return cfg

    def get_runner_code_and_logs(self):
        logs = "\n".join(OLLAMA_LOG_PATH.read_text(errors="replace").splitlines()[-80:]) if OLLAMA_LOG_PATH.exists() else ""
        return {"script_path": str(RUNNER_SCRIPT_PATH), "config_path": str(RUNNER_CONFIG_PATH), "log_path": str(OLLAMA_LOG_PATH), "script": RUNNER_SCRIPT_PATH.read_text() if RUNNER_SCRIPT_PATH.exists() else "", "config": json.loads(RUNNER_CONFIG_PATH.read_text()) if RUNNER_CONFIG_PATH.exists() else {}, "logs": logs, "last_updated_at": self.last_updated_script_at}

    def is_ollama_installed(self): return shutil.which("ollama") is not None

    async def is_ollama_running(self):
        try:
            async with httpx.AsyncClient(timeout=1.5) as c:
                return (await c.get(f"{OLLAMA_HOST}/api/version")).status_code == 200
        except Exception: return False

    async def get_ollama_version(self):
        try:
            async with httpx.AsyncClient(timeout=2.0) as c:
                r = await c.get(f"{OLLAMA_HOST}/api/version")
                if r.status_code == 200: return r.json().get("version")
        except Exception: pass
        return None

    async def get_installed_models(self):
        try:
            async with httpx.AsyncClient(timeout=3.0) as c:
                r = await c.get(f"{OLLAMA_HOST}/api/tags")
                if r.status_code == 200:
                    return [InstalledModel(name=m["name"], size_bytes=m.get("size", 0), size_gb=round(m.get("size", 0)/(1024**3), 2), modified_at=m.get("modified_at"), family=(m.get("details") or {}).get("family"), parameter_size=(m.get("details") or {}).get("parameter_size"), quantization_level=(m.get("details") or {}).get("quantization_level")) for m in r.json().get("models", [])]
        except Exception: pass
        return []

    async def get_loaded_models(self):
        try:
            async with httpx.AsyncClient(timeout=2.0) as c:
                r = await c.get(f"{OLLAMA_HOST}/api/ps")
                if r.status_code == 200: return [m["name"] for m in r.json().get("models", [])]
        except Exception: pass
        return []

    async def stop_ollama(self):
        subprocess.run(["pkill", "-f", "ollama serve"], check=False)
        self.state = "idle"
        self.state_message = "Ollama server stopped"
        return {"stopped": True}

    async def delete_model(self, name: str):
        async with httpx.AsyncClient(timeout=10.0) as c:
            r = await c.request("DELETE", f"{OLLAMA_HOST}/api/delete", json={"name": name})
            return {"deleted": r.status_code == 200, "model": name}

    async def deploy_and_start(self, req: DeployRequest):
        emit = lambda step, msg, prog=None, **kw: json.dumps({"step": step, "message": msg, "progress": prog, **kw}) + "\n"
        try:
            self.state = "configuring"
            cfg = self._write_runner_files(req, initial=False)
            yield emit("code_updated", f"Updated Colab server runner script for {req.model} (Ollama {req.ollama_version})", 15, config=cfg)
            cur_ver = await self.get_ollama_version()
            if not self.is_ollama_installed() or (req.ollama_version != "latest" and cur_ver and req.ollama_version not in cur_ver):
                self.state = "starting"
                yield emit("installing_ollama", f"Installing Ollama ({req.ollama_version}) on Google Colab...", 25)
                env = os.environ.copy()
                if req.ollama_version != "latest": env["OLLAMA_VERSION"] = req.ollama_version
                p = await asyncio.create_subprocess_shell("curl -fsSL https://ollama.com/install.sh | sh", env=env, stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.STDOUT)
                async for raw in p.stdout:
                    line = raw.decode(errors="ignore").strip()
                    if line: yield emit("install_log", line, 30)
                await p.wait()
            if await self.is_ollama_running() and req.restart_server:
                await self.stop_ollama()
                await asyncio.sleep(0.8)
            if not await self.is_ollama_running():
                self.state = "starting"
                yield emit("starting_daemon", "Starting `ollama serve` on Colab GPU...", 42)
                env = os.environ.copy()
                env.update({"OLLAMA_HOST": "127.0.0.1:11434", "OLLAMA_ORIGINS": "*", "OLLAMA_KEEP_ALIVE": req.keep_alive, "OLLAMA_FLASH_ATTENTION": "1" if req.flash_attention else "0"})
                lf = open(OLLAMA_LOG_PATH, "a")
                self.process = subprocess.Popen(["ollama", "serve"], env=env, stdout=lf, stderr=subprocess.STDOUT, start_new_session=True)
                for _ in range(30):
                    if await self.is_ollama_running(): break
                    await asyncio.sleep(0.5)
            self.state = "pulling"
            yield emit("pull_start", f"Pulling '{req.model}' onto Colab GPU...", 52)
            last_ts = 0.0
            async with httpx.AsyncClient(timeout=None) as c:
                async with c.stream("POST", f"{OLLAMA_HOST}/api/pull", json={"name": req.model, "stream": True}) as r:
                    async for line in r.aiter_lines():
                        if not line.strip(): continue
                        evt = json.loads(line)
                        if "error" in evt:
                            self.state = "error"
                            yield emit("error", evt["error"], 52)
                            return
                        tot, done = evt.get("total", 0), evt.get("completed", 0)
                        now = time.time()
                        if tot and (now - last_ts >= 0.25 or done >= tot):
                            last_ts = now
                            pct = round((done / tot) * 100, 1)
                            yield emit("pull_progress", f"{evt.get('status', 'Downloading')}: {round(done/1048576, 1)} / {round(tot/1048576, 1)} MB ({pct}%)", int(52 + (done/tot)*40), download_percent=pct)
            yield emit("warming_up", f"Loading '{req.model}' into GPU VRAM...", 95)
            async with httpx.AsyncClient(timeout=90.0) as c:
                await c.post(f"{OLLAMA_HOST}/api/generate", json={"model": req.model, "prompt": "", "keep_alive": req.keep_alive})
            self.active_model = req.model
            self.state = "ready"
            self.state_message = f"Active & serving {req.model}"
            yield emit("ready", f"Colab server updated and serving '{req.model}' via ngrok!", 100, active_model=req.model)
        except Exception as e:
            self.state = "error"
            yield emit("error", str(e), 0)

    async def stream_chat(self, req: ChatRequest):
        target = req.model or self.active_model
        async with httpx.AsyncClient(timeout=None) as c:
            async with c.stream("POST", f"{OLLAMA_HOST}/api/chat", json={"model": target, "messages": [m.model_dump() for m in req.messages], "stream": True, "options": {"temperature": req.temperature, "num_ctx": req.num_ctx}}) as r:
                async for line in r.aiter_lines():
                    if line.strip(): yield line + "\n"
''')

(SERVER_PKG / "main.py").write_text('''import os, time
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from .catalog_scraper import fetch_ollama_library_catalog
from .gpu_monitor import get_gpu_metrics, get_system_metrics
from .models import ChatRequest, DeployRequest, ServerStatusResponse
from .ollama_manager import OllamaManager

app = FastAPI(title="Ollama Cloud Runner Colab Backend")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
manager = OllamaManager()

@app.get("/api/status", response_model=ServerStatusResponse)
async def status():
    running = await manager.is_ollama_running()
    ver = await manager.get_ollama_version()
    inst = await manager.get_installed_models() if running else []
    loaded = await manager.get_loaded_models() if running else []
    active = manager.active_model or (loaded[0] if loaded else None)
    return ServerStatusResponse(connected=True, state=manager.state, state_message=manager.state_message, ollama_installed=manager.is_ollama_installed(), ollama_running=running, ollama_version=ver, configured_ollama_version=manager.configured_ollama_version, active_model=active, loaded_models=loaded, installed_models=inst, gpu=get_gpu_metrics(), system=get_system_metrics(), ngrok_url=os.environ.get("PUBLIC_NGROK_URL"), uptime_seconds=int(time.time()-manager.started_at), last_updated_script_at=manager.last_updated_script_at)

@app.get("/api/models/catalog")
async def catalog():
    cat = await fetch_ollama_library_catalog()
    inst = await manager.get_installed_models()
    bases = {m.name.split(":")[0] for m in inst}
    return {"count": len(cat), "installed": [m.model_dump() for m in inst], "models": [{**m, "is_installed": m["name"] in bases} for m in cat]}

@app.get("/api/server/code")
async def code(): return manager.get_runner_code_and_logs()

@app.post("/api/server/deploy")
async def deploy(req: DeployRequest):
    return StreamingResponse(manager.deploy_and_start(req), media_type="application/x-ndjson")

@app.post("/api/server/stop")
async def stop(): return await manager.stop_ollama()

@app.delete("/api/models/{model_name:path}")
async def delete_model(model_name: str): return await manager.delete_model(model_name)

@app.post("/api/chat")
async def chat(req: ChatRequest):
    return StreamingResponse(manager.stream_chat(req), media_type="application/x-ndjson")
''')

print("✅ Modular Colab backend package created at /content/ollama_cloud_runner/server")

In [ ]:
# @title Step 3: Start Colab Server & Expose via Ngrok
import asyncio
import getpass
import os
import uvicorn
from pyngrok import conf, ngrok

ngrok_token = ""
try:
    from google.colab import userdata
    ngrok_token = userdata.get("NGROK_AUTHTOKEN") or ""
except Exception:
    pass

if not ngrok_token:
    ngrok_token = getpass.getpass("🔑 Paste your NGROK_AUTHTOKEN (from dashboard.ngrok.com): ").strip()

conf.get_default().auth_token = ngrok_token
ngrok.kill()
tunnel = ngrok.connect(addr=8000, proto="http")
os.environ["PUBLIC_NGROK_URL"] = tunnel.public_url

print("\n" + "=" * 74)
print("🚀 GOOGLE COLAB BACKEND CONNECTED!")
print(f"🔗 Paste this Ngrok URL into your Web App:  {tunnel.public_url}")
print("=" * 74 + "\n")

config = uvicorn.Config("server.main:app", host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()